# MyBasket – RF-DETR Nano POC
Entraînement Colab gratuit, isolé de la production MyBasket.

In [ ]:
%pip -q install "rfdetr[train,onnx]" supervision pillow onnx onnxruntime


In [ ]:
import os, json, shutil, zipfile, torch
from pathlib import Path
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ABSENT')
if not torch.cuda.is_available():
    raise RuntimeError('Active un GPU dans Colab : Exécution > Modifier le type d’exécution > GPU.')


## Charger le zip du dataset
Le zip doit contenir `train/`, `valid/`, `test/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
zip_name=next(name for name in uploaded if name.lower().endswith('.zip'))
root=Path('/content/mybasket_dataset')
if root.exists(): shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(zip_name) as z: z.extractall(root)
candidates=[p.parent for p in root.rglob('train/_annotations.coco.json')]
if not candidates: raise FileNotFoundError('train/_annotations.coco.json introuvable')
DATASET=candidates[0]
for split in ('train','valid','test'):
    data=json.loads((DATASET/split/'_annotations.coco.json').read_text())
    print(split, len(data['images']), 'images', len(data['annotations']), 'annotations')


## Entraînement RF-DETR Nano

In [ ]:
from rfdetr import RFDETRNano
model=RFDETRNano()
model.train(dataset_dir=str(DATASET), epochs=60, batch_size='auto', lr=1e-4,
            output_dir='/content/mybasket_rfdetr_output', early_stopping=True,
            early_stopping_patience=10)


## Checkpoint et contrôle visuel

In [ ]:
ckpts=list(Path('/content/mybasket_rfdetr_output').rglob('*.pth'))
if not ckpts: raise FileNotFoundError('Aucun checkpoint .pth trouvé')
BEST=max(ckpts,key=lambda p:p.stat().st_mtime)
print('Checkpoint retenu:',BEST)


In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
eval_model=RFDETRNano(pretrain_weights=str(BEST))
test_json=json.loads((DATASET/'test'/'_annotations.coco.json').read_text())
classes={c['id']-1:c['name'] for c in test_json['categories']}
for rec in test_json['images'][:12]:
    img=Image.open(DATASET/'test'/rec['file_name']).convert('RGB')
    det=eval_model.predict(img,threshold=0.30)
    draw=ImageDraw.Draw(img)
    for box,score,cid in zip(det.xyxy,det.confidence,det.class_id):
        x1,y1,x2,y2=map(float,box)
        draw.rectangle((x1,y1,x2,y2),outline='red',width=3)
        draw.text((x1,max(0,y1-14)),f"{classes.get(int(cid),cid)} {float(score):.2f}",fill='red')
    plt.figure(figsize=(8,8)); plt.imshow(img); plt.axis('off'); plt.show()


## Export ONNX et téléchargement

In [ ]:
export_model=RFDETRNano(pretrain_weights=str(BEST))
onnx_path=export_model.export(output_dir='/content/mybasket_onnx',format='onnx')
print('ONNX:',onnx_path)
from google.colab import files
files.download(str(BEST))
onnx_files=list(Path('/content/mybasket_onnx').rglob('*.onnx'))
if not onnx_files: raise FileNotFoundError('Export ONNX introuvable')
files.download(str(onnx_files[0]))
